### Mesaages

Messages are fundamental unit of context for models in langchain. They represent the input and output of models, carrying both the content and metadata needed to represent the conversation when iterating with LLM. Message are object that contain:

Role: Identifies the message type (e.g. System & User)
Content: Represent the actual content of the message (Like text, Images, audio, document etc)
Metadata: Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message Type that works across all model providers, ensuring consistent behaviour regardless of the model being called.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key.startswith("sk-"):
    print("successfully loaded OpenAI API key")
else:
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
if anthropic_api_key.startswith("sk"):
      print("successfully loaded Anthropic API key")
else:
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")

if google_api_key.startswith("AI"):
    print("successfully loaded Google API key")
else:
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
if deepseek_api_key.startswith("sk"):
    print("successfully loaded DeepSeek API key")
else:
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
if groq_api_key.startswith("gsk"):
    print("successfully loaded GROQ API key")
else:
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
if grok_api_key.startswith("xai-"):
    print("successfully loaded GROK API key")
else:
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
if openrouter_api_key.startswith("sk-"):
    print("successfully loaded OpenRouter API key")
else:
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
if not openai_api_key and not anthropic_api_key and not google_api_key and not deepseek_api_key and not groq_api_key and not grok_api_key and not openrouter_api_key:
    print("No API keys were found. Please check your .env file and ensure you have the correct keys set.")

from langchain.chat_models import init_chat_model
os.environ["openrouter_api_key"] = openrouter_api_key
model = init_chat_model(
    "openrouter:deepseek/deepseek-v4-flash",)

In [ ]:
model.invoke("What is the artifical intelligence?")

### Text Prompts

Text prompts are strings - ideal for straight forword generation tasks where you don't need to retain conversation history.
Text prompts are the most basic way to interact with a language model. You simply provide a string of text as input, and the model generates a response based on that input. This is often used for simple tasks like asking questions or generating text.

In [ ]:
model.invoke("What is langchain?")

Use text prompt when:
- you have a single, stand alone request
- you don't need conversation history
- you want minimal code complexity

### Message Prompts

Alternatively, you can pass in list of messages to the model by providing a list of message objects.

Message Types
- System message - Tells to model how to behave and provide the context for interactions
- Human messages - Represent user input and interact with the model
- AI messages    - Responses generated by the model, including text content, tool calls and meatadata
- Tool messages  - Represent the output of tool calls 

### System messages
A system message represent an initial set of instruction that primes the model's behavior. You can use a system message to set the tone, define the model's role, and establish guidelines for response.

 ### Human messages
 A human message represent user input and interactions. They can contain text, images, audio files and other amount of multimodel content


### AI Message
An AI message represent the output of model invocation. They can include multimodel data, tool calls and provider specific metadata that you can later access

### Tool Message
For model that support tool calling, AI message can contain tool calls. Tool messages are used to pass the results of single tool execution back to the model

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
messages = [
    SystemMessage("You are a helpful assistant that translates English to French."), 
    HumanMessage("Translate this sentence from English to French. I love programming."),
    #AIMessage("J'adore programmer.")
]
response = model.invoke(messages)
response.content

In [ ]:
system_msg = SystemMessage("You are a helpful coding assistant")
messages = [
    system_msg,
    HumanMessage("tell me different types of HTTP requests")
]
response = model.invoke(messages)
print(response.content)

In [ ]:
## Detailed information to the LLM through the SystemMessage. The LLM will use this information to answer the HumanMessage.

from langchain.messages import SystemMessage, HumanMessage

system_msg = SystemMessage("""You are a senior pyton developer with expertise in web frameworks. always provide code examples and explain your resoning." \
"be consise but throrough in your explanations""")

messages = [
  system_msg,
  HumanMessage("tell me different types of HTTP requests"),
]
response = model.invoke(messages)
print(response.content)

In [ ]:
### Messages metadata
human_msg = HumanMessage(
  content="Hello!",
  name="greeting", #optional: identify the different users in a conversation
  id="12345", #optional: unique identifier for the message    
)

In [ ]:
response = model.invoke([human_msg])
response.content

In [ ]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage
# create AI message manually
AI_msg = AIMessage("Hello! How can I assist you today?")

# Add to conversation history
messages = [
    SystemMessage("You are a helpful assistant"), 
    HumanMessage("Can you help me?"),
    AI_msg,
    HumanMessage("Yes, what is 2+2?"),
]
response = model.invoke(messages)
response.content

In [ ]:
#usage metadata is used to track the number of tokens used in the conversation, which can be useful for monitoring costs and usage limits. You can access the usage metadata from the response object after invoking the model.
response.usage_metadata


In [ ]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage
# after model makes a tool call
# (Here, we demonstrate manually creating a message for brevity
# in practice, the model would generate this message automatically after making a tool call

ai_message = AIMessage(
  content=[],
  tool_calls=[{
    "name":"get_weather",
    "args":{"location":"New York"},
    "id":"call 12345"
  }]
)

# execute the tool call and get the result
weather_result = "The weather in New York is sunny and 75 degrees."
tool_message = ToolMessage(
  content=weather_result,
  tool_call_id="call 12345"
)
# continue the conversation with the tool result
messages = [
  HumanMessage("What's the weather like in san francisco?"),
  ai_message,
  tool_message
]

response = model.invoke(messages)
print(response.content)